# Distributed Training Launcher

Notebook version of `run_distributed.sh`. Each cell uses `%%bash`, so the logic
runs exactly as it would in the shell script. This launches `train.py` — it does
**not** contain `train.py`'s own training logic (see `train_cifar10.ipynb` for that).

Run cells top to bottom. The **Config** cell (in place of CLI args) controls
epochs / batch size / lr / tracker.

⚠️ **Heads up on "distributed":** `train.py` doesn't currently wrap the model in
`DistributedDataParallel` or shard data across ranks, so even when this launches
via `torchrun` on multiple GPUs, each process trains an independent full copy of
the model rather than participating in real distributed training. This notebook
reproduces the launcher faithfully as-is — it doesn't fix that gap.

## GPU Detection

In [ ]:
%%bash
echo "===================================="
echo "Distributed Training Launcher"
echo "===================================="

if command -v nvidia-smi >/dev/null 2>&1; then
    GPU_COUNT=$(nvidia-smi -L | wc -l)
else
    GPU_COUNT=0
fi

echo "Detected GPUs: $GPU_COUNT"
echo "$GPU_COUNT" > /tmp/gpu_count.txt

_Note: `%%bash` cells run in their own subshell, so `GPU_COUNT` doesn't persist to later cells on its own — it's written to `/tmp/gpu_count.txt` here and re-read in the Launch cell below._

## Config

Replaces the `--epochs`, `--batch-size`, `--lr`, `--tracker` CLI flags from the
original script. Edit these values directly instead of passing arguments.

This cell is plain Python (not `%%bash`) so the values can be reused to build the
shell commands in later cells via environment variables.

In [ ]:
import os

EPOCHS = 10
BATCH_SIZE = 32
LR = 0.001
TRACKER = "none"

# Pass into the bash cells below via environment variables
os.environ["EPOCHS"] = str(EPOCHS)
os.environ["BATCH_SIZE"] = str(BATCH_SIZE)
os.environ["LR"] = str(LR)
os.environ["TRACKER"] = str(TRACKER)

print(f"Epochs: {EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Learning Rate: {LR}")
print(f"Tracker: {TRACKER}")

## Validation

Same checks as the script (`epochs > 0`, `batch-size > 0`). Run this after editing
Config above — it will raise and stop the notebook if either value is invalid,
same as `exit 1` did in the script.

In [ ]:
assert EPOCHS > 0, "Error: epochs must be > 0"
assert BATCH_SIZE > 0, "Error: batch-size must be > 0"
print("Validation passed.")

## Logging Setup

Creates the timestamped run directory, same as the script.

In [ ]:
%%bash
RUN_ID=$(date +"%Y%m%d_%H%M%S")
LOG_DIR="runs/run_$RUN_ID"

mkdir -p "$LOG_DIR"

echo "Run Directory: $LOG_DIR"
echo "Epochs: $EPOCHS"
echo "Batch Size: $BATCH_SIZE"
echo "Learning Rate: $LR"
echo "Tracker: $TRACKER"

# persist LOG_DIR for the Launch cell, since %%bash cells don't share state
echo "$LOG_DIR" > /tmp/log_dir.txt

_Note: like `GPU_COUNT`, `LOG_DIR` is written to a temp file (`/tmp/log_dir.txt`) so the Launch cell below can pick it up — `%%bash` cells don't persist variables to each other the way notebook Python cells do._

## Launch

Mirrors the script's branch: multi-GPU runs go through `torchrun`, otherwise it
falls back to plain `python train.py`. Make sure `train.py` is in the same
working directory as this notebook (or update the path below).

In [ ]:
%%bash
GPU_COUNT=$(cat /tmp/gpu_count.txt)
LOG_DIR=$(cat /tmp/log_dir.txt)

if [ "$GPU_COUNT" -gt 1 ]; then

    echo "Launching torchrun (multi-GPU)"

    torchrun \
        --nproc_per_node=$GPU_COUNT \
        train.py \
        --epochs $EPOCHS \
        --batch-size $BATCH_SIZE \
        --lr $LR \
        --output-dir $LOG_DIR

else

    echo "Single GPU / CPU fallback"

    python train.py \
        --epochs $EPOCHS \
        --batch-size $BATCH_SIZE \
        --lr $LR \
        --output-dir $LOG_DIR

fi

echo "Run Complete"